In [ ]:
# etl_analysis.py
# Simple analysis for churn_data with displayed charts

import os
import pandas as pd
import matplotlib.pyplot as plt
from supabase import create_client
from dotenv import load_dotenv


# Connect to Supabase
def get_supabase_client():
    load_dotenv()
    url = os.getenv("SUPABASE_URL")
    key = os.getenv("SUPABASE_KEY")
    return create_client(url, key)


# Load table
def load_churn_data():
    client = get_supabase_client()
    response = client.table("churn_data").select("*").execute()
    data = response.data if hasattr(response, "data") else response.get("data", [])
    return pd.DataFrame(data)


# Main
def run_analysis():
    df = load_churn_data()

    if df.empty:
        print("No data found.")
        return

    # Handle base directory for saving CSV and charts
    try:
        base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    except:
        base_dir = os.getcwd()

    processed_dir = os.path.join(base_dir, "data", "processed")
    os.makedirs(processed_dir, exist_ok=True)

    # ---- METRICS ----

    churn_percentage = (df["Churn"] == "Yes").mean() * 100

    avg_monthly = df.groupby("Contract")["MonthlyCharges"].mean()

    tenure_counts = df["tenure_group"].value_counts()

    internet_counts = df["InternetService"].value_counts()

    pivot_table = pd.crosstab(df["tenure_group"], df["Churn"])

    # ---- PRINT CLEAN SUMMARY ----

    print("\n--- Analysis Summary ---\n")

    print("Churn Percentage:", round(churn_percentage, 2), "%\n")

    print("Average Monthly Charges by Contract:")
    print(avg_monthly, "\n")

    print("Tenure Group Counts:")
    print(tenure_counts, "\n")

    print("Internet Service Distribution:")
    print(internet_counts, "\n")

    print("Churn vs Tenure Group:")
    print(pivot_table, "\n")

    # ---- SAVE SUMMARY CSV ----

    summary = {
        "churn_percentage": round(churn_percentage, 2),
        "avg_monthly_month_to_month": avg_monthly.get("Month-to-month", None),
        "avg_monthly_one_year": avg_monthly.get("One year", None),
        "avg_monthly_two_year": avg_monthly.get("Two year", None),
        "new_customers": tenure_counts.get("New", 0),
        "regular_customers": tenure_counts.get("Regular", 0),
        "loyal_customers": tenure_counts.get("Loyal", 0),
        "champion_customers": tenure_counts.get("Champion", 0),
    }

    summary_path = os.path.join(processed_dir, "analysis_summary.csv")
    pd.DataFrame([summary]).to_csv(summary_path, index=False)

    print("Summary saved to:", summary_path)

    # ---- CHARTS (Displayed) ----

    # 1. Churn rate by Monthly Charge Segment
    df.groupby("monthly_charge_segment")["Churn"] \
      .apply(lambda x: (x == "Yes").mean()) \
      .plot(kind="bar")
    plt.title("Churn Rate by Monthly Charge Segment")
    plt.show()

    # 2. Histogram of Total Charges
    df["TotalCharges"].plot(kind="hist", bins=30)
    plt.title("Total Charges Histogram")
    plt.show()

    # 3. Contract type distribution
    df["Contract"].value_counts().plot(kind="bar")
    plt.title("Contract Type Distribution")
    plt.show()

    print("Analysis complete.")


# Run
if __name__ == "__main__":
    run_analysis()


Reading churn_data table...

--- Analysis Summary ---

Churn Percentage: 25.6 %

Average Monthly Charges by Contract:
Contract
Month-to-month    67.297996
One year          68.184069
Two year          63.269838
Name: MonthlyCharges, dtype: float64

Tenure Group Counts:
tenure_group
New         321
Regular     268
Champion    209
Loyal       202
Name: count, dtype: int64

Internet Service Distribution:
InternetService
Fiber optic    468
DSL            329
No             203
Name: count, dtype: int64

Churn vs Tenure Group:
Churn          No  Yes
tenure_group          
Champion      196   13
Loyal         169   33
New           167  154
Regular       212   56

Summary saved to: c:\AIDS Tekworks\TELECONNECT\data\processed\analysis_summary.csv
Creating charts...
Charts created.
Analysis complete.


<Figure size 640x480 with 0 Axes>